In [1]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Resolve absolute paths to local Silver Delta table directories
PROJECT_ROOT = os.path.dirname(os.getcwd())

TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_kafka_stream")
TARGET_DLQ_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "silver_kafka_stream_dlq")

# Initialize SparkSession with Delta support
builder = (
    SparkSession.builder.appName("Query_Silver")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/Users/intuitivecode/Code/deltastreams/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/intuitivecode/.ivy2/cache
The jars for the packages stored in: /Users/intuitivecode/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e6fc3270-6959-4429-9431-fa729a27e200;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 87ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default    

In [2]:
silver_df = spark.read.format("delta").load(TARGET_PATH)
print(f"Total Clean Network Telemetry Records: {silver_df.count():,}")
silver_df.select("host", "src_ip", "dst_ip", "protocol", "src_port", "dst_port", "_processed_at").show(10, truncate=False)

26/10/18 13:28:40 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/10/18 13:28:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total Clean Network Telemetry Records: 111
+----------+-------------+--------+--------+--------+--------+-----------------------+
|host      |src_ip       |dst_ip  |protocol|src_port|dst_port|_processed_at          |
+----------+-------------+--------+--------+--------+--------+-----------------------+
|172.21.0.1|             |        |        |NULL    |NULL    |2026-10-17 17:46:14.265|
|172.21.0.1|192.168.1.100|10.0.0.5|TCP     |443     |5140    |2026-10-17 17:46:14.265|
|172.21.0.1|             |        |        |NULL    |NULL    |2026-10-17 17:46:14.265|
|172.21.0.1|192.168.1.100|10.0.0.5|TCP     |443     |5140    |2026-10-17 17:46:14.265|
|172.21.0.1|192.168.1.100|10.0.0.5|TCP     |443     |5140    |2026-10-17 17:46:14.265|
|172.21.0.1|             |        |        |NULL    |NULL    |2026-10-17 17:46:14.265|
|172.21.0.1|             |        |        |NULL    |NULL    |2026-10-17 17:46:14.265|
|172.21.0.1|192.168.1.100|10.0.0.5|TCP     |443     |5140    |2026-10-17 17:46:14.265|


In [ ]:
# Load Silver DLQ Delta table
dlq_df = spark.read.format("delta").load(TARGET_DLQ_PATH)

print(f"Total Quarantined DLQ Records: {dlq_df.count()}")
dlq_df.select("event_time", "event_type", "price", "_dlq_reason", "_source_file").show(10, truncate=False)

In [13]:
# KAFKA STREAMS
dlq_df = spark.read.format("delta").load(TARGET_DLQ_PATH)

print(f"Total Quarantined Network Logs (DLQ): {dlq_df.count():,}")
dlq_df.select("host", "message", "_dlq_reason", "_processed_at").show(10, truncate=False)


Total Quarantined Network Logs (DLQ): 156
+----------+------------------------------------------------------------------+-------------------------+----------------------+
|host      |message                                                           |_dlq_reason              |_processed_at         |
+----------+------------------------------------------------------------------+-------------------------+----------------------+
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_7761527361_NULL_EXCEPTION !!!|MALFORMED_UDP_BYTE_STREAM|2026-10-17 14:34:00.08|
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_7761527361_NULL_EXCEPTION !!!|MALFORMED_UDP_BYTE_STREAM|2026-10-17 14:34:00.08|
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_7761527361_NULL_EXCEPTION !!!|MALFORMED_UDP_BYTE_STREAM|2026-10-17 14:34:00.08|
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_7761527361_NULL_EXCEPTION !!!|MALFORMED_UDP_BYTE_STREAM|2026-10-17 14:34:00.08|
|172.20.0.1|!!! CORRUPT_BYTE_STREAM_INVALID_UTF8_776152